# Cloud Removal from Satellite Imagery - Training Notebook

This notebook trains a **Pix2Pix GAN** (U-Net Generator + PatchGAN Discriminator) to remove clouds from optical satellite images using the **RICE1 & RICE2** datasets.

**Pipeline:** Clone repo -> Install deps -> Preprocess data -> Train on GPU -> Track with DagsHub/MLflow

> Make sure to set **Runtime > Change runtime type > GPU (T4)** before running.

## 1. Setup: Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/Bhar07gavi/Cloud_Removal.git
%cd Cloud_Removal

In [ ]:
# Install dependencies
!pip install -q torch torchvision dagshub 'dvc[s3]' mlflow pyyaml tqdm matplotlib scikit-image Pillow numpy

In [ ]:
# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Dataset: Pull via DVC or Preprocess Locally

In [ ]:
# Option A: Pull preprocessed data via DVC (if configured with DagsHub)
# Uncomment the lines below after configuring DagsHub DVC remote
# !dvc pull -r dagshub

# Option B: Download RICE dataset and run preprocessing locally
# If the dataset directory already exists (e.g., pushed via DVC), skip download.
import os
if not os.path.exists("dataset/RICE1/cloud"):
    print("Dataset not found. Please upload or download the RICE dataset first.")
    print("Expected structure: dataset/RICE1/cloud/, dataset/RICE1/label/,")
    print("                   dataset/RICE2/cloud/, dataset/RICE2/label/")
else:
    print("Dataset found! Running preprocessing...")
    !python -m src.data.preprocess --config configs/dataset.yaml

## 3. (Optional) DagsHub + MLflow Authentication

In [ ]:
# Uncomment and fill in your DagsHub credentials to enable MLflow tracking
# import dagshub
# dagshub.init(
#     repo_owner="YOUR_DAGSHUB_USERNAME",
#     repo_name="Cloud_Removal",
#     mlflow=True
# )
#
# # After dagshub.init(), the MLflow tracking URI is set automatically.
# import mlflow
# print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

## 4. Run Smoke Test (Sanity Check)

In [ ]:
# Quick verification that models and losses work correctly
!python tests/smoke_test.py

## 5. Train the Model

Launch training with GPU acceleration. Adjust `--epochs` as needed.
Default config: `batch_size=16`, `lr=0.0002`, `lambda_l1=100`.

In [ ]:
# Launch training (adjust epochs as needed)
!python -m src.training.train --config configs/train.yaml --epochs 50

## 6. View Training Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Display the latest sample grid
sample_dir = Path("checkpoints/sample_images")
if sample_dir.exists():
    images = sorted(sample_dir.glob("*.png"))
    if images:
        latest = images[-1]
        print(f"Showing: {latest.name}")
        img = mpimg.imread(str(latest))
        plt.figure(figsize=(15, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Sample Grid: {latest.name}")
        plt.show()
    else:
        print("No sample images found yet.")
else:
    print("Checkpoints directory not found. Has training started?")

In [ ]:
# List saved checkpoints
import os
ckpt_dir = "checkpoints"
if os.path.exists(ckpt_dir):
    files = sorted(os.listdir(ckpt_dir))
    print("Saved checkpoints:")
    for f in files:
        if f.endswith(".pt"):
            size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
            print(f"  {f} ({size_mb:.1f} MB)")

## 7. View MLflow Dashboard (DagsHub)

If you configured DagsHub in Step 3, view your live metrics at:

`https://dagshub.com/<YOUR_USERNAME>/Cloud_Removal/experiments`